In [1]:
from models import *
from algorithms import *

import pandas
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer
import wandb
from torch.utils.data import DataLoader
import os
import numpy as np
import time
import tqdm

os.environ['WANDB_NOTEBOOK_NAME'] = 'model_training.ipynb'
wandb.login()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

wandb: Currently logged in as: giacomoaru (giacomo-aru). Use `wandb login --relogin` to force relogin


In [2]:
print('device in use:', torch.cuda.get_device_name(device))

device in use: NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
df = pandas.read_csv("../data/training_split_proc.csv", sep=";").sample(frac=1, random_state=69)
training_len = int(len(df)*0.8)
training_set = df.iloc[:training_len]
validation_set = df.iloc[training_len:]

training_len = len(training_set)
validation_len = len(validation_set)

max_token_len = 256

model_name = 'FacebookAI/xlm-roberta-base'
# model_name = '/opt/models/FacebookAI/xlm-roberta-base'

In [4]:
# class to create a custom "DataSet" object, compatible with pytorch
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer):
        self.len = len(dataframe)

        # tokenized dataframe
        self.input_values = [tokenizer(a, padding="max_length", max_length=max_token_len,  return_tensors='pt', truncation=True) for a in dataframe["processed_tweet"].values]
        # gold labels
        self.labels = torch.from_numpy(dataframe[['NO', 'IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE', 
                                                  'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']].values.astype(np.float32))

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        input_values = self.input_values[idx]
        labels = self.labels[idx]

        return input_values, labels

In [5]:
# correct tokenizer releted to the selected pretrained transformer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# training and validation sets are refactored as "CustomDataset" objects
tr_data = CustomDataset(training_set, tokenizer=tokenizer)
val_data = CustomDataset(validation_set, tokenizer=tokenizer)

A custom "collate function" (link) is required in order to format data in the correct way. This function reformat batches, and is exploited by the DataLoader Pytorch object to return, during iteration, data shaped as expected.

In [6]:
def my_collate_fn(batch):
    
    # tokenized samples (inputs and targets) are grouped in batches
    input = {'input_ids':torch.stack(([x[0]['input_ids'][0] for x in batch])).to(device), 'attention_mask':torch.stack(([x[0]['attention_mask'][0] for x in batch])).to(device)}
    labels = torch.stack(([x[1] for x in batch])).to(device)
        
    return [input, labels]

In [7]:
# function to call to save the model in a specific path, file name must be .pt
def save_model(model:nn.Module, path:str):
    torch.save(model.state_dict(), path)

# load a saved model (MultiLabelClassifier class)
# config necessary to reproduce the same nn architecture (pretrained model, hidden size, output size ecc...)
def load_model(path:str, config:dict): 
    if config['class'] == 'not_chain':
        classifier = MultiLabelClassifier(config['pretrained_name'], config['hidden_layer_size'])
    else:
        classifier = MultiLabelClassifierChain(config['pretrained_name'], config['hidden_layer_size'])
    
    classifier.load_state_dict(torch.load(path))

    return classifier

In [8]:
verbose = True

In [9]:
def train_one_epoch(epoch, tot_batch, tr_data_loader, n_split, classifier, optimizer, criterion):
    tr_loss = 0.0
    epoch_time_start = time.time()
    
    # reset of gradients
    optimizer.zero_grad() 
    
    with tqdm.tqdm(total=tot_batch, desc=f'epoch {epoch}') as pbar:
        
        batch_loss = 0
        for i, batch in enumerate(tr_data_loader):
                    
            input, labels = batch
                    
            # setting the timer for calculating statistics
            batch_time_start = time.time()

            # Forward Phase
            if isinstance(classifier, MultiLabelClassifierChain):
                output = classifier(input, labels[:,:1])
            else:
                output = classifier(input)
                
            loss = criterion(output, labels)
            batch_loss += loss.item()
            tr_loss += loss.item()
                    
            # Backward Phase - Gradient Accumulation
            loss.backward()
            
            # Whenever the gradient of a real mini-batch of data is accumulated, a learning step is performed
            if (i + 1)%n_split == 0:
                
                optimizer.step()
                
                # reset of gradients
                optimizer.zero_grad() 
                
                batch_time = time.time() - batch_time_start
                wandb.log({'tr_batch_loss': batch_loss/n_split, 'batch_time':batch_time})  
                batch_loss = 0
                
                pbar.update(1)
        
    return tr_loss/(i + 1), time.time() - epoch_time_start

In [10]:
def build_optimizer(network, optimizer, learning_rate):
    if optimizer == "sgd":
        optimizer = optim.SGD(network.parameters(),
                              lr=learning_rate, momentum=0.9)
    elif optimizer == "adam":
        optimizer = optim.Adam(network.parameters(),
                               lr=learning_rate)
    elif optimizer == "adamw":
        optimizer = optim.AdamW(network.parameters(),
                               lr=learning_rate)
    return optimizer

In [11]:
def train(config=None):
    with wandb.init(config=config):
        
        config = wandb.config
        config.pretrained_name = 'xlm-roberta-base'
        
        mean_epoch_time = 0
        tot_batch = int(training_len/config.batch_size)
        
        # maximum batch size supportable by the GPU of the current device
        # !!!!!!!!!! supportable_batch_size MUST BE A DIVISOR OF config.batch_size !!!!!!!!!!
        supportable_batch_size = 1
        
        supportable_batch_size = min(supportable_batch_size, config.batch_size)
        n_split = int(config.batch_size/supportable_batch_size)
        
        if verbose: print(n_split, "shots of", supportable_batch_size, 
              "elements  -  config batch_size:", config.batch_size)
        
        torch.manual_seed(config.weight_seed)
        if config.classifier_type == 'chain':
            classifier = MultiLabelClassifierChain(config.pretrained_name, 
                                                   config.dropout, 
                                                   config.hidden_layer_size,
                                                   int(config.hidden_layer_size*(1/2))).to(device)
        else:
            classifier = MultiLabelClassifier(config.pretrained_name, 
                                              config.dropout, 
                                              config.hidden_layer_size, 
                                              int(config.hidden_layer_size*(1/2))).to(device)
        classifier.freeze_pretrained()
        criterion = nn.BCELoss()
        optimizer = build_optimizer(classifier, config.optimizer, config.learning_rate)
        
        # "DataLoader" objects are exploited to iterate on the dataset in batches
        torch.manual_seed(config.data_seed)
        tr_data_loader = DataLoader(tr_data, batch_size=supportable_batch_size, shuffle=True, collate_fn=my_collate_fn)
        val_data_loader = DataLoader(val_data, batch_size=supportable_batch_size, shuffle=True, collate_fn=my_collate_fn)
        
        for epoch in range(config.epochs): 
            
            if epoch >= config.finetune_from_epoch:
                classifier.unfreeze_pretrained()
                
            # model modules switch their behaviour to "training mode"
            classifier.train()
            tr_loss, epoch_time = train_one_epoch(epoch, tot_batch, tr_data_loader, n_split, classifier, optimizer, criterion)
            mean_epoch_time += epoch_time
            
            # model modules switch their behaviour to "evaluation mode"
            classifier.eval()
            # context-manager that disables gradient calculation
            with torch.no_grad():
                
                # validation error is calculated in parts using a batch mechanism to control memory usage
                val_loss = 0
                for j, (input, labels) in enumerate(val_data_loader):
                    
                    if isinstance(classifier, MultiLabelClassifierChain):
                        output = classifier(input, labels[:,:1])
                    else:
                        output = classifier(input)
                    
                    val_loss += criterion(output, labels).item()
                val_loss = val_loss/(j + 1)
                
                hard_avg, hard_classes, soft_avg = evaluate_model(classifier, val_data_loader)
                wandb.log(hard_avg)
                wandb.log(soft_avg)
                wandb.log(hard_classes)
                
                wandb.log({'tr_loss': tr_loss, 'val_loss':val_loss, 'epoch':epoch})   
                if verbose: print(f'ep time: {epoch_time:.1f} tr: {tr_loss:.6f} vl: {val_loss:.6f}')
                
        mean_epoch_time /= epoch + 1
        wandb.log({'mean_epoch_time':mean_epoch_time})
        # save_model(classifier, '../data/models/tmp.pt')

In [12]:
class c:
    def __init__(self):
        self.learning_rate = 0.01
        self.batch_size = 32
        self.epochs = 15
        self.optimizer = 'adam'
        self.hidden_layer_size = 1024
        
        self.freeze_pretrained = True
        
        self.dropout = 0.25
        self.classifier_type = 'chain'
config = c()

In [13]:
wandb.agent('hltproject/EXIST2024/1efrvjbd', train, count=1)
wandb.finish()
# train(config)

2024-05-01 11:25:12,059 - wandb.agents.pyagent - INFO -                  run() - Starting sweep agent: entity=None, project=None, count=1


wandb: Agent Starting Run: 6rcgeq3w with config:
wandb: 	batch_size: 64
wandb: 	classifier_type: chain
wandb: 	data_seed: 9
wandb: 	dropout: 0.15000000000000002
wandb: 	epochs: 10
wandb: 	finetune_from_epoch: 10
wandb: 	hidden_layer_size: 128
wandb: 	learning_rate: 0.0013956093944433518
wandb: 	optimizer: adamw
wandb: 	weight_seed: 69
wandb: Currently logged in as: giacomoaru (hltproject). Use `wandb login --relogin` to force relogin


64 shots of 1 elements  -  config batch_size: 64


epoch 0:  99%|█████████▉| 79/79.5625 [01:57<00:00,  1.48s/it]


2024-05-01 11:28:10,925 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'FMeasure']
2024-05-01 11:28:11,175 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2024-05-01 11:28:11,904 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
cargado 24
2024-05-01 11:28:12,591 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft']
2024-05-01 11:28:13,025 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method
ep time: 117.1 tr: 0.448573 vl: 0.422489


epoch 1:  41%|████▏     | 33/79.5625 [00:49<01:18,  1.69s/it]